# Notebook 1: Setup del Entorno y Exploración del Dataset

**Trabajo Práctico – Visión por Computadora II**  
**Tema:** Sistema de Monitoreo Automático de EPP en Obras de Construcción  

---

## Objetivos de este notebook
1. Verificar el entorno (Python, CUDA/CPU, dependencias)
2. Descargar el dataset Construction-PPE desde Roboflow Universe
3. Realizar un Análisis Exploratorio de Datos (EDA)
4. Visualizar muestras y distribución de clases

## 1. Verificación del Entorno

In [ ]:
import sys
import platform

print(f"Python: {sys.version}")
print(f"Sistema: {platform.system()} {platform.machine()}")

# Verificar PyTorch y dispositivo disponible
try:
    import torch
    print(f"\nPyTorch: {torch.__version__}")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Dispositivo: {device.upper()}")
    if device == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("⚠️  CUDA no disponible – entrenando en CPU (más lento, se recomienda reducir epochs)")
except ImportError:
    print("PyTorch no instalado. Ejecutar: uv pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu")

In [ ]:
# Verificar Ultralytics
import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")
ultralytics.checks()

## 2. Descarga del Dataset desde Roboflow

Dataset: **Construction Site Safety** (Roboflow Universe)  
Clases: `Hardhat`, `Mask`, `NO-Hardhat`, `NO-Mask`, `NO-Safety Vest`, `Person`, `Safety Cone`, `Safety Vest`, `machinery`, `vehicle`

> **Nota:** Necesitás una API key de Roboflow (gratuita).  
> Registrate en https://roboflow.com y copiá tu key desde Account → API Keys

In [ ]:
import os
from pathlib import Path

# ======================================================
# CONFIGURACIÓN – completar con tu API key de Roboflow
# ======================================================
ROBOFLOW_API_KEY = "TU_API_KEY_AQUI"   # <--- Reemplazar
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Directorio de datos: {DATA_DIR.resolve()}")

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Dataset: Construction Site Safety
# https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(28)   # versión estable con YOLOv8 format

dataset = version.download("yolov8", location=str(DATA_DIR / "construction-ppe"))
print(f"\nDataset descargado en: {dataset.location}")

In [ ]:
# Verificar estructura del dataset
import os

dataset_path = Path(dataset.location)
for split in ['train', 'valid', 'test']:
    split_dir = dataset_path / split / 'images'
    if split_dir.exists():
        n_images = len(list(split_dir.glob('*.jpg'))) + len(list(split_dir.glob('*.png')))
        print(f"  {split:6s}: {n_images:5d} imágenes")

# Leer el archivo data.yaml
import yaml
with open(dataset_path / 'data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)

print(f"\nClases ({data_config['nc']}):", data_config['names'])

## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Distribución de Clases

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid", palette="husl")

def count_class_distribution(labels_dir: Path, class_names: list) -> Counter:
    """Cuenta instancias de cada clase en un directorio de labels YOLO."""
    counter = Counter()
    for label_file in labels_dir.glob("*.txt"):
        with open(label_file) as f:
            for line in f:
                cls_id = int(line.strip().split()[0])
                counter[class_names[cls_id]] += 1
    return counter

class_names = data_config['names']

# Contar por split
splits_counts = {}
for split in ['train', 'valid', 'test']:
    labels_dir = dataset_path / split / 'labels'
    if labels_dir.exists():
        splits_counts[split] = count_class_distribution(labels_dir, class_names)

# Visualización
fig, axes = plt.subplots(1, len(splits_counts), figsize=(18, 5))
if len(splits_counts) == 1:
    axes = [axes]

colors = sns.color_palette("husl", len(class_names))

for ax, (split, counts) in zip(axes, splits_counts.items()):
    sorted_counts = dict(sorted(counts.items(), key=lambda x: x[1], reverse=True))
    bars = ax.barh(list(sorted_counts.keys()), list(sorted_counts.values()), color=colors)
    ax.set_title(f"Split: {split} ({sum(counts.values())} instancias)", fontsize=13, fontweight='bold')
    ax.set_xlabel("Cantidad de instancias")
    for bar, val in zip(bars, sorted_counts.values()):
        ax.text(val + 20, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)

plt.suptitle("Distribución de Clases por Split – Construction-PPE Dataset", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../data/class_distribution.png", bbox_inches='tight', dpi=150)
plt.show()
print("Figura guardada en data/class_distribution.png")

### 3.2 Distribución de Tamaños de Bounding Boxes

In [ ]:
def get_bbox_stats(labels_dir: Path):
    """Extrae w, h normalizados de todos los bboxes."""
    widths, heights, areas = [], [], []
    for label_file in labels_dir.glob("*.txt"):
        with open(label_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    w, h = float(parts[3]), float(parts[4])
                    widths.append(w)
                    heights.append(h)
                    areas.append(w * h)
    return widths, heights, areas

train_labels = dataset_path / 'train' / 'labels'
widths, heights, areas = get_bbox_stats(train_labels)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(widths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title("Distribución de Ancho (w normalizado)")
axes[0].set_xlabel("Ancho relativo"); axes[0].set_ylabel("Frecuencia")

axes[1].hist(heights, bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title("Distribución de Alto (h normalizado)")
axes[1].set_xlabel("Alto relativo")

axes[2].hist(areas, bins=50, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[2].set_title("Distribución de Área (w×h)")
axes[2].set_xlabel("Área relativa")

plt.suptitle(f"Estadísticas de Bounding Boxes – Train ({len(widths):,} instancias)", fontsize=13)
plt.tight_layout()
plt.savefig("../data/bbox_stats.png", bbox_inches='tight', dpi=150)
plt.show()

print(f"\nEstadísticas de áreas:")
print(f"  Mediana: {np.median(areas):.4f}")
print(f"  Media:   {np.mean(areas):.4f}")
print(f"  Mín:     {np.min(areas):.4f}")
print(f"  Máx:     {np.max(areas):.4f}")

### 3.3 Visualización de Imágenes con Anotaciones

In [ ]:
import cv2
import random
from PIL import Image

# Paleta de colores por clase (EPP presente = verde, ausente = rojo)
COLORS = {
    'Hardhat':        (0, 200, 0),
    'Mask':           (0, 180, 0),
    'Safety Vest':    (0, 160, 0),
    'NO-Hardhat':     (220, 0, 0),
    'NO-Mask':        (200, 0, 0),
    'NO-Safety Vest': (180, 0, 0),
    'Person':         (100, 100, 255),
    'Safety Cone':    (255, 165, 0),
    'machinery':      (128, 0, 128),
    'vehicle':        (64, 64, 64),
}

def draw_yolo_annotations(image_path: Path, label_path: Path, class_names: list) -> np.ndarray:
    """Dibuja bboxes YOLO sobre una imagen."""
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                cls_name = class_names[cls_id]
                color = COLORS.get(cls_name, (200, 200, 200))
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, cls_name, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

# Seleccionar imágenes random del train
train_images = list((dataset_path / 'train' / 'images').glob('*.jpg'))
random.seed(42)
sample_images = random.sample(train_images, min(8, len(train_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, img_path in enumerate(sample_images):
    label_path = dataset_path / 'train' / 'labels' / (img_path.stem + '.txt')
    img = draw_yolo_annotations(img_path, label_path, class_names)
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(img_path.name[:30], fontsize=8)

plt.suptitle("Muestras del Dataset con Anotaciones (Verde=EPP presente, Rojo=EPP ausente)", 
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../data/sample_annotations.png", bbox_inches='tight', dpi=150)
plt.show()
print("Figura guardada en data/sample_annotations.png")

## 4. Resumen del Dataset y Preparación para Entrenamiento

In [ ]:
import pandas as pd

# Tabla resumen
summary = []
for split in ['train', 'valid', 'test']:
    images_dir = dataset_path / split / 'images'
    labels_dir = dataset_path / split / 'labels'
    if images_dir.exists():
        n_imgs = len(list(images_dir.glob('*.jpg'))) + len(list(images_dir.glob('*.png')))
        counts = count_class_distribution(labels_dir, class_names)
        total_annots = sum(counts.values())
        summary.append({'Split': split, 'Imágenes': n_imgs, 'Anotaciones totales': total_annots,
                        'Anotaciones/imagen': round(total_annots/n_imgs, 1)})

df_summary = pd.DataFrame(summary)
print("=== Resumen del Dataset ===")
print(df_summary.to_string(index=False))

# Guardar ruta del dataset para los notebooks siguientes
DATASET_YAML = str(dataset_path / 'data.yaml')
print(f"\n✅ Dataset YAML para entrenamiento: {DATASET_YAML}")
print("   → Copiar esta ruta para usarla en el Notebook 2")

In [ ]:
# Guardar la ruta en un archivo de configuración compartido
import json

config = {"dataset_yaml": DATASET_YAML, "num_classes": data_config['nc'], "class_names": class_names}
with open("../data/dataset_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Configuración guardada en data/dataset_config.json")
print(json.dumps(config, indent=2))

---
## ✅ Checklist Notebook 1
- [ ] Entorno verificado (Python, PyTorch, Ultralytics)
- [ ] Dataset descargado desde Roboflow
- [ ] Distribución de clases analizada
- [ ] Estadísticas de bboxes calculadas
- [ ] Muestras visualizadas con anotaciones
- [ ] dataset_config.json guardado

**Próximo paso:** Notebook 2 – Entrenamiento de Modelos YOLO